# txtai Market Research — End-to-End Example

Walks through the full project pipeline against AAPL:

1. Ingest a small batch of SEC filings + news articles
2. Confirm the documents land in the txtai index
3. Drive the agentic research pipeline (router -> retrievers -> synthesizer)
4. Inspect routing, retrievals, and the synthesized answer

Prerequisites: `docker-compose up -d` must be running so KeyDB,
PostgreSQL+pgvector, and MinIO are reachable. Secrets must be present in
`.env` (`SEC_USER_AGENT`, `NEWSAPI_KEY`, `ALPHAVANTAGE_API_KEY`,
`OPENAI_API_KEY`).

## Endpoints exercised in this notebook

The notebook touches three layers of the project. Knowing which call
lives where makes it easy to swap in a different driver (CLI, HTTP,
Streamlit) for the same pipeline.

### Python entry points (used directly by this notebook)

| Symbol                                       | Source                              | Purpose                                                  | Cell |
|----------------------------------------------|-------------------------------------|----------------------------------------------------------|------|
| `app.collectors.SECCollector.collect(...)`   | `app/collectors/sec_collector.py`   | Pull filings into MinIO + Postgres + txtai               |  1   |
| `app.pipeline.embeddings.get_embeddings()`   | `app/pipeline/embeddings.py`        | Singleton `txtai.Embeddings` index used by every agent   |  2   |
| `app.pipeline.embeddings.search(query, k)`   | `app/pipeline/embeddings.py`        | Thin wrapper around `Embeddings.search`                  |  2   |
| `app.agents.research_agent.run_research_sync(query)` | `app/agents/research_agent.py` | Sync route -> retrieve -> synthesize, returns one dict   |  3   |
| `app.agents.research_agent.run_research(query)`      | `app/agents/research_agent.py` | Generator form, yields one event per pipeline stage      |  4   |

### HTTP endpoints (the same pipeline, exposed for UIs)

Defined in `app/api/server.py` — not invoked from this notebook, but the
request bodies match the function signatures above.

| Verb / Path             | Body                  | Behavior                                                                |
|-------------------------|-----------------------|-------------------------------------------------------------------------|
| `GET  /`                | -                     | Health probe, returns the registered endpoint map                       |
| `POST /research`        | `{"query": "..."}`    | Calls `run_research_sync`, returns a JSON dict with the full trace      |
| `POST /research/stream` | `{"query": "..."}`    | SSE stream of `route` -> `retrieve` -> `synthesize` -> `done` events    |

### txtai primitives under the hood

Every retrieval cell ultimately resolves to a `txtai.Embeddings` call:

- `Embeddings.search(query, limit)` for plain semantic top-k
- `Embeddings.search("... where tags = 'sec'", parameters=..., limit=...)`
  for per-source scoping inside the SEC and News sub-agents
- `Embeddings.save(path)` / `.load(path)` for the on-disk index in `data/`
- Optional `txtai.LLM(model)` in the synthesizer when LLM credentials are set

See `notebooks/txtai.API.ipynb` for each of those primitives in isolation.

In [1]:
%load_ext autoreload
%autoreload 2

# System libraries.
import logging
import os
import sys
from pathlib import Path

# Third party libraries.
from dotenv import load_dotenv

In [2]:
# Make the project root importable when running the notebook from notebooks/.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
# Load secrets from .env and configure notebook logging.
load_dotenv(project_root / ".env")
_LOG = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
_LOG.info("OPENAI_API_KEY present: %s", bool(os.getenv("OPENAI_API_KEY")))
_LOG.info("SEC_USER_AGENT present: %s", bool(os.getenv("SEC_USER_AGENT")))

INFO __main__: OPENAI_API_KEY present: False
INFO __main__: SEC_USER_AGENT present: True


## 1. Ingest a small batch

Pull a handful of recent SEC filings for AAPL into MinIO + PostgreSQL +
the txtai index. Limit kept small so the cell finishes in a couple of
minutes.

In [4]:
# Run the SEC collector for one ticker.
import importlib, app.collectors.sec_collector as sc                                                                                                                                                               
importlib.reload(sc) 
from app.collectors import SECCollector

sec = SECCollector()
sec_summary = sec.collect(
    ticker="AAPL",
    filing_types=["10-K", "8-K"],
    limit=2,
)
_LOG.info("SEC collection summary: %s", sec_summary)

/Users/gprakash/src/umd_classes1/class_project/data605/Spring2026/projects/UmdTask430_DATA605_Spring2026_txtai_for_market_research/.venv/lib/python3.14/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
INFO faiss.loader: Loading faiss.
INFO faiss.loader: Successfully loaded faiss.
INFO app.storage.cold_storage.minio_client: MinIO client created for endpoint: localhost:9000
INFO app.storage.cold_storage.minio_client: Connected to MinIO at localhost:9000
INFO app.storage.warm_storage.pgvector_client: PostgreSQL connection pool created: min=1, max=10
INFO app.storage.warm_storage.pgvector_client: Connected to PostgreSQL at localhost:5432/financial_kb
INFO app.storage.hot_storage.keydb_client: Connected to Key

## 2. Confirm the index has content

`get_embeddings()` returns the singleton index used by every agent.

In [5]:
# Spot-check the index by issuing one semantic query.
from app.pipeline.embeddings import get_embeddings, search

embeddings = get_embeddings()
_LOG.info("Index row count: %d", embeddings.count())
hits = search("Apple revenue trend", limit=3)
for h in hits:
    _LOG.info("score=%.3f text=%s", h["score"], (h.get("text") or "")[:120])

INFO __main__: Index row count: 7
INFO __main__: score=0.516 text=The information contained in this Current Report shall not be deemed “filed” for purposes of Section 18 of the Securitie
INFO __main__: score=0.498 text=aapl-20260430 false 0000320193 0000320193 2026-04-30 2026-04-30 0000320193 us-gaap:CommonStockMember 2026-04-30 2026-04-
INFO __main__: score=0.495 text=Date: April 20, 2026 
 

 
 Apple Inc. 
 

 

 

   

   

   

 

 

   

 
 By: 
 

 
 /s/ Jennifer Newstead 
 
 

 




## 3. Run the agentic pipeline synchronously

`run_research_sync` is the same entry point the FastAPI `/research`
handler calls. It returns a single dict with the full trace.

In [6]:
# Drive the full route -> retrieve -> synthesize pipeline.
from app.agents.research_agent import run_research_sync

result = run_research_sync("What are the key risks discussed in Apple's latest 10-K?")
_LOG.info("Route: %s", result.get("route"))
_LOG.info("Used LLM: %s", result.get("used_llm"))
_LOG.info("Chunk count: %d", result.get("chunk_count", 0))
print(result.get("answer", ""))

INFO app.agents.research_agent: SEC agent query=What are the key risks discussed in Apple's latest 10-K? ticker=AAPL
INFO __main__: Route: {'query': "What are the key risks discussed in Apple's latest 10-K?", 'ticker': 'AAPL', 'agents': ['sec'], 'reason': 'Question mentions SEC / filings keywords; routing to SEC agent only.', 'elapsed_ms': 0.07258300320245326, 'step_ms': 0.06799999391660094}
INFO __main__: Used LLM: False
INFO __main__: Chunk count: 5


☐ Item 5.02 Departure of Directors or Certain Officers; Election of Directors; Appointment of Certain Officers; Compensatory Arrangements of Certain Officers . On April 20, 2026, Apple Inc. (“Apple”) announced that Tim Cook will transition from his role as Chief Executive Officer to Executive Chair of Apple’s Board of Directors (the “Board”), effective September 1, 2026 (the “Transition Date”). [1] The information contained in this Current Report shall not be deemed “filed” for purposes of Section 18 of the Securities Exchange Act of 1934, as amended (the “Exchange Act”), or incorporated by reference in any filing under the Securities Act of 1933, as amended, or the Exchange Act, except as shall be expressly set forth by specific reference in such a filing. Item 9.01 Financial Statements and Exhibits. (d) Exhibits. [2] aapl-20260430 false 0000320193 0000320193 2026-04-30 2026-04-30 0000320193 us-gaap:CommonStockMember 2026-04-30 2026-04-30 0000320193 aapl:A1.625NotesDue2026Member 2026-

## 4. Stream the same query

`run_research` is the generator form. The FastAPI SSE endpoint and the
Streamlit UI consume this so they can show each stage as it happens.

In [7]:
# Iterate over the streaming events and log each one as it arrives.
from app.agents.research_agent import run_research

for event in run_research("Any analyst upgrades for AAPL recently?"):
    step = event["step"]
    if step == "synthesize":
        _LOG.info("[%s] answer=%s", step, event["payload"].get("answer", "")[:160])
    elif step == "retrieve":
        chunks = event["payload"].get("chunks", [])
        _LOG.info("[%s] agent=%s chunks=%d", step, event["payload"].get("agent"), len(chunks))
    else:
        _LOG.info("[%s] payload keys=%s", step, list(event["payload"].keys()))

INFO __main__: [route] payload keys=['query', 'status', 'elapsed_ms']
INFO __main__: [route] payload keys=['query', 'ticker', 'agents', 'reason', 'elapsed_ms', 'step_ms']
INFO __main__: [retrieve] agent=news chunks=0
INFO app.agents.research_agent: News agent query=Any analyst upgrades for AAPL recently? ticker=AAPL
INFO __main__: [retrieve] agent=news chunks=0
INFO __main__: [synthesize] answer=
INFO __main__: [synthesize] answer=I couldn't find relevant documents about **AAPL**. Searched the news index. Try rephrasing or asking about a different ticker (we have data for AAPL, MSFT, NVDA
INFO __main__: [done] payload keys=['chunk_count', 'timings']


## Summary

- The full ingest -> index -> search -> agent pipeline runs end-to-end
  from a notebook with no API server required
- The same `run_research_sync` function powers the FastAPI handler and
  the Streamlit UI; this notebook is the primary smoke test for the
  research pipeline
- To extend: add another collector (`app/collectors/my_collector.py`),
  re-run the ingest cell, and the agent picks up the new chunks
  automatically because they share the txtai index